In [0]:
%run ./common_utils

In [0]:
CAT = 'machine_health.prod'
AUDIT = CAT + '.pipeline_run_audit'
declare_widgets()
mt = dbutils.widgets.get('machine_type')
with audited() as a:
    maybe_disk_fault()
    ev = (spark.table(f'{CAT}.predictions_latest')
          .select('sensor_id', 'reading_time', 'axis', 'vibration_rms_mm_s'))
    new = (spark.table(f'{CAT}.alert_candidates').where(scope(mt))
           .join(ev, ['sensor_id', 'reading_time', 'axis'])
           .select('sensor_id', 'machine_type', 'reading_time',
                   F.lit('HIGH_VIBRATION').alias('category'),
                   F.lit('Schedule inspection').alias('advice'),
                   F.concat(F.lit('vibration_rms_mm_s='),
                            F.col('vibration_rms_mm_s').cast('string')).alias('details')))
    seen = spark.table(f'{CAT}.maintenance_advice').select('sensor_id', 'reading_time')
    (new.join(seen, ['sensor_id', 'reading_time'], 'left_anti')
        .write.mode('append').saveAsTable(f'{CAT}.maintenance_advice'))

/home/spark-8d8e0835-6151-4073-bc0d-91/.ipykernel/63/command-7089605854406250-1338218180:94: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time=datetime.utcnow()
/home/spark-8d8e0835-6151-4073-bc0d-91/.ipykernel/63/command-7089605854406250-1338218180:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  rec['end_time'] = datetime.utcnow()
